<a href="https://colab.research.google.com/github/dream001871-cell/xcxc/blob/main/pytorch%E7%A4%BA%E4%BE%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# -*- coding: utf-8 -*-
import torch  # 导入PyTorch核心库
import math   # 导入数学库，用于生成pi等常量

# 生成输入x：在[-π, π]区间内生成2000个均匀分布的点
x = torch.linspace(-math.pi, math.pi, 2000)
# 生成真实标签y：对每个x计算sin(x)（这是我们要拟合的目标）
y = torch.sin(x)

In [20]:
# 定义幂次：1,2,3（对应x^1, x^2, x^3）
p = torch.tensor([1, 2, 3])
# 构造特征矩阵xx：shape从(2000,) → (2000, 3)
xx = x.unsqueeze(-1).pow(p)

In [21]:
# 定义模型：Sequential是“顺序容器”，按顺序执行层操作
model = torch.nn.Sequential(
    # 线性层：输入维度3（对应xx的3个特征），输出维度1
    torch.nn.Linear(3, 1),
    # 展平层：将输出从(2000, 1)展平为(2000,)，匹配y的shape
    torch.nn.Flatten(0, 1)
)

In [22]:
# 损失函数：均方误差（MSE），reduction='sum'表示求和（而非求平均）
loss_fn = torch.nn.MSELoss(reduction='sum')

In [23]:
learning_rate = 1e-6  # 学习率：控制参数更新的步长
for t in range(2000):  # 训练2000轮

    # 1. 前向传播：用模型预测y_pred
    y_pred = model(xx)

    # 2. 计算损失：对比预测值和真实值
    loss = loss_fn(y_pred, y)
    # 每100轮打印一次损失值（看训练进度）
    if t % 100 == 99:
        print(t, loss.item())

    # 3. 清零梯度：PyTorch梯度会累加，必须先清零
    model.zero_grad()

    # 4. 反向传播：计算损失对所有可学习参数的梯度
    loss.backward()

    # 5. 梯度下降更新参数：手动更新（无优化器）
    with torch.no_grad():  # 该上下文内，张量不会计算梯度（节省资源）
        for param in model.parameters():  # 遍历模型所有可学习参数（weight/bias）
            param -= learning_rate * param.grad  # 核心更新公式：参数 = 参数 - 学习率×梯度

99 693.6503295898438
199 466.4495849609375
299 314.8138122558594
399 213.55450439453125
499 145.8961181640625
599 100.66096496582031
699 70.39823913574219
799 50.138755798339844
899 36.56645965576172
999 27.467334747314453
1099 21.36264419555664
1199 17.263561248779297
1299 14.509020805358887
1399 12.656424522399902
1499 11.409309387207031
1599 10.569025039672852
1699 10.002335548400879
1799 9.619792938232422
1899 9.361300468444824
1999 9.186456680297852


In [24]:
# 取出模型的第一个层（线性层）
linear_layer = model[0]

# 打印最终的多项式公式
print(f'Result: y = {linear_layer.bias.item()} + {linear_layer.weight[:, 0].item()} x + {linear_layer.weight[:, 1].item()} x^2 + {linear_layer.weight[:, 2].item()} x^3')

Result: y = 0.012726453132927418 + 0.8421682119369507 x + -0.0021955247502774 x^2 + -0.09125755727291107 x^3


用 PyTorch 内置的 RMSprop 优化器替代了手动的梯度更新逻辑，整体目标依然是用三次多项式 y=b+w1​x+w2​x2+w3​x3 拟合正弦函数

In [25]:
learning_rate = 1e-3  # 学习率提升到1e-3（RMSprop适配性更好）
# 定义RMSprop优化器：传入模型参数 + 学习率
optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

In [26]:
for t in range(2000):  # 训练2000轮
    # 1. 前向传播：预测y_pred
    y_pred = model(xx)

    # 2. 计算损失
    loss = loss_fn(y_pred, y)
    if t % 100 == 99:
        print(t, loss.item())  # 每100轮打印损失

    # 3. 清零梯度（优化器层面清零）
    optimizer.zero_grad()

    # 4. 反向传播：计算梯度
    loss.backward()

    # 5. 更新参数（优化器自动完成）
    optimizer.step()

99 8.817180633544922
199 8.817537307739258
299 9.359833717346191
399 8.893904685974121
499 8.979092597961426
599 8.988024711608887
699 8.911643981933594
799 8.900290489196777
899 8.921662330627441
999 8.929132461547852
1099 8.919719696044922
1199 8.91787338256836
1299 8.921265602111816
1399 8.921805381774902
1499 8.92051887512207
1599 8.920397758483887
1699 8.920883178710938
1799 8.920896530151367
1899 8.920702934265137
1999 8.920731544494629


In [27]:
# 取出线性层
linear_layer = model[0]
# 打印拟合的多项式公式
print(f'Result: y = {linear_layer.bias.item()} + {linear_layer.weight[:, 0].item()} x + {linear_layer.weight[:, 1].item()} x^2 + {linear_layer.weight[:, 2].item()} x^3')

Result: y = -0.0005000155069865286 + 0.8572406768798828 x + -0.0005000189412385225 x^2 + -0.09283044934272766 x^3


PyTorch 中自定义神经网络模块，核心目标依然是用三次多项式 y=a+bx+cx2+dx3 拟合正弦函数 sin(x)，但和前两版的核心区别是：不再用 nn.Sequential 拼接现成层，而是通过继承 torch.nn.Module 自定义完整的模型类，更贴近实际项目中复杂模型的定义方式。

通过自定义 Polynomial3 类（继承 nn.Module）定义三次多项式模型，手动声明可学习参数（a/b/c/d），实现前向传播逻辑；然后用 MSE 损失函数和 SGD 优化器训练模型，最终得到拟合 sin(x) 的多项式参数。相比前两版，这是更 “原生” 的 PyTorch 模型定义方式，灵活性更高（适合复杂模型）。

In [28]:
class Polynomial3(torch.nn.Module):
    def __init__(self):
        """构造函数：初始化模型的可学习参数"""
        super().__init__()  # 必须调用父类nn.Module的构造函数
        # 定义4个可学习参数（标量），用nn.Parameter包裹（关键！）
        self.a = torch.nn.Parameter(torch.randn(()))
        self.b = torch.nn.Parameter(torch.randn(()))
        self.c = torch.nn.Parameter(torch.randn(()))
        self.d = torch.nn.Parameter(torch.randn(()))

    def forward(self, x):
        """前向传播函数：定义模型的计算逻辑（必须实现）"""
        return self.a + self.b * x + self.c * x ** 2 + self.d * x ** 3

    def string(self):
        """自定义方法：格式化输出模型参数（非必需，仅方便打印）"""
        return f'y = {self.a.item()} + {self.b.item()} x + {self.c.item()} x^2 + {self.d.item()} x^3'

（1）继承 nn.Module 的意义
torch.nn.Module 是 PyTorch 所有模型的基类，提供了核心功能：

自动管理可学习参数（nn.Parameter 标记的张量）；
支持前向传播（forward 方法）和反向传播（自动微分）；
兼容优化器（model.parameters() 可获取所有参数）。

（2）nn.Parameter 关键作用

用 nn.Parameter 包裹的张量会被标记为「可学习参数」，model.parameters() 能自动收集这些参数，供优化器更新；
如果直接定义 self.a = torch.randn(())（不加 nn.Parameter），这个张量不会被优化器识别，训练时不会更新；
torch.randn(())：生成一个标准正态分布的标量张量（shape: []），作为参数的初始值。

（3）forward 方法

必须实现的核心方法，定义 “输入 x → 输出 y_pred” 的计算逻辑；
这里直接按三次多项式公式计算：y=a+bx+cx2+dx3，无需像前两版那样构造 x2/x3 特征矩阵，更直观；
当调用 model(x) 时，PyTorch 会自动执行 forward(x) 方法（无需手动调用）。

（4）自定义 string 方法
仅为了方便格式化打印最终的多项式公式，属于辅助功能，不是模型必需的。

In [29]:
x = torch.linspace(-math.pi, math.pi, 2000)
y = torch.sin(x)

In [30]:
# 实例化自定义模型
model = Polynomial3()

# 损失函数：MSE（求和模式），和前两版一致
criterion = torch.nn.MSELoss(reduction='sum')
# 优化器：SGD（随机梯度下降），传入模型所有可学习参数
optimizer = torch.optim.SGD(model.parameters(), lr=1e-6)

In [31]:
for t in range(2000):  # 训练2000轮
    # 1. 前向传播：调用model(x)自动执行forward(x)，得到预测值
    y_pred = model(x)

    # 2. 计算损失：对比预测值和真实值
    loss = criterion(y_pred, y)
    # 每100轮打印损失（观察训练进度）
    if t % 100 == 99:
        print(t, loss.item())

    # 3. 清零梯度：避免梯度累加
    optimizer.zero_grad()

    # 4. 反向传播：自动计算损失对所有参数（a/b/c/d）的梯度
    loss.backward()

    # 5. 更新参数：SGD优化器自动调整a/b/c/d的值
    optimizer.step()
print(f'Result: {model.string()}')

99 143.37506103515625
199 99.24335479736328
299 69.63803100585938
399 49.76138687133789
499 36.40550994873047
599 27.423725128173828
699 21.37820816040039
799 17.30529022216797
899 14.558857917785645
999 12.705070495605469
1099 11.452587127685547
1199 10.605522155761719
1299 10.032052040100098
1399 9.643392562866211
1499 9.379692077636719
1599 9.200592041015625
1699 9.078814506530762
1799 8.995917320251465
1899 8.939425468444824
1999 8.90088176727295
Result: y = 0.00700585450977087 + 0.8505999445915222 x + -0.001208627363666892 x^2 + -0.09245689958333969 x^3


动态计算图，核心目标依然是拟合正弦函数 sin(x)，但模型的前向传播逻辑是动态变化的（每一次前向传播的计算路径都可能不同），这展示了 PyTorch 最核心的特性之一：动态图机制（运行时构建计算图）。

In [36]:
import random
class DynamicNet(torch.nn.Module):

    def __init__(self):
        """构造函数：初始化5个可学习参数（标量）"""
        super().__init__()
        self.a = torch.nn.Parameter(torch.randn(()))
        self.b = torch.nn.Parameter(torch.randn(()))
        self.c = torch.nn.Parameter(torch.randn(()))
        self.d = torch.nn.Parameter(torch.randn(()))
        self.e = torch.nn.Parameter(torch.randn(()))  # 复用的参数（x^4/x^5共享）

    def forward(self, x):
        """
        前向传播：动态计算路径（核心！）
        1. 基础部分：三次多项式 a + bx + cx² + dx³
        2. 动态部分：随机选择叠加 x^4（仅） 或 x^4+x^5（均用参数e）
        """
        # 基础三次多项式计算
        y = self.a + self.b * x + self.c * x ** 2 + self.d * x ** 3
        # 随机生成上限：4或5 → 循环范围是 4到上限（不包含）
        # 情况1：randint(4,6)=4 → 循环范围4~4（不执行）→ 仅三次多项式
        # 情况2：randint(4,6)=5 → 循环范围4~5 → 执行1次（exp=4）→ 加e*x^4
        # 情况3：randint(4,6)=6 → 循环范围4~6 → 执行2次（exp=4/5）→ 加e*x^4 + e*x^5
        for exp in range(4, random.randint(4, 6)):
            y = y + self.e * x ** exp
        return y

    def string(self):
        """自定义方法：格式化输出参数（标注e的复用）"""
        return f'y = {self.a.item()} + {self.b.item()} x + {self.c.item()} x^2 + {self.d.item()} x^3 + {self.e.item()} x^4 ? + {self.e.item()} x^5 ?'

In [37]:
# 实例化动态模型
model = DynamicNet()

# 损失函数：MSE（求和模式）
criterion = torch.nn.MSELoss(reduction='sum')
# 优化器：带动量的SGD（关键！适配动态模型的难训练特性）
optimizer = torch.optim.SGD(model.parameters(), lr=1e-8, momentum=0.9)

In [38]:
for t in range(30000):  # 迭代3万次（远多于前几版的2千次）
    # 1. 前向传播：每次调用model(x)都可能生成不同的计算图
    y_pred = model(x)

    # 2. 计算损失
    loss = criterion(y_pred, y)
    # 每2000轮打印一次损失（迭代次数多，降低打印频率）
    if t % 2000 == 1999:
        print(t, loss.item())

    # 3. 清零梯度
    optimizer.zero_grad()

    # 4. 反向传播：根据本次的动态计算图，自动计算所有参数的梯度
    loss.backward()

    # 5. 更新参数：带动量的SGD自动调整参数
    optimizer.step()

1999 1465.160888671875
3999 701.6158447265625
5999 315.91375732421875
7999 146.78292846679688
9999 69.53044128417969
11999 35.77273941040039
13999 20.800647735595703
15999 14.098953247070312
17999 11.140422821044922
19999 9.777207374572754
21999 9.271103858947754
23999 9.017290115356445
25999 8.734903335571289
27999 8.872335433959961
29999 8.595451354980469
